# 因子测试框架应用示例 - 518880 3月数据分析

这个notebook演示如何使用因子测试框架对518880的3月数据进行分析，计算IC、IR、分组测试等指标。

## 1. 环境设置和导入

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# 设置显示选项
pd.set_option('display.max_rows', 20)
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 150)

# 设置绘图样式
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# 添加路径
sys.path.append('/home/jovyan/base_demo')
sys.path.append('/home/jovyan/work/tactics_demo')
sys.path.append('/home/jovyan/work/tactics_demo/factor_testing')

## 2. 导入因子测试框架

In [3]:
# 导入因子测试框架
from factor_testing import FactorData, FactorPreprocessor, FactorMetrics, GroupTester, ReportGenerator

# 导入delta模块用于特征提取
from delta import FeatureExtractor, create_feature, get_trade_dates, split_dates_by_range

# 导入基础工具
import base_tool

print("所有模块导入成功!")

所有模块导入成功!


## 3. 数据准备和特征提取

In [4]:
# 设置参数
instrument_id = '518880'
start_date = '20260301'
end_date = '20260331'

# 特征提取参数
param_dict = {
    'instrument_id': instrument_id,
    'short_window': 60,
    'long_window': 300,
    'stride': 1
}

print(f"分析标的: {instrument_id}")
print(f"分析期间: {start_date} 到 {end_date}")

分析标的: 518880
分析期间: 20260301 到 20260331


In [5]:
# 获取交易日
all_trade_dates = get_trade_dates()

# 筛选3月数据
march_dates = [date for date in all_trade_dates 
               if date >= start_date and date <= end_date]

print(f"3月总交易日数量: {len(march_dates)}")
print(f"交易日: {march_dates[:5]}...{march_dates[-5:] if len(march_dates) > 10 else ''}")

3月总交易日数量: 20
交易日: ['20260302', '20260303', '20260304', '20260305', '20260306']...['20260323', '20260324', '20260325', '20260326', '20260327']


## 4. 特征提取函数

In [6]:
def extract_features_for_date(trade_ymd, instrument_id, param_dict):
    """提取单日特征"""
    try:
        # 加载数据
        snap_list = base_tool.snap_list_load(instrument_id, trade_ymd)
        if not snap_list:
            return None
        
        # 提取特征
        features_list = []
        times_list = []
        
        short_window = param_dict.get('short_window', 60)
        stride = param_dict.get('stride', 1)
        
        # 滑动窗口提取特征
        for i in range(short_window, len(snap_list), stride):
            snap_slice = snap_list[i-short_window:i]
            features = create_feature(snap_slice, short_window)
            features_list.append(features)
            times_list.append(snap_list[i]['time_mark'])
        
        # 转换为DataFrame
        if features_list:
            df = pd.DataFrame(features_list)
            df['time_mark'] = times_list
            df['date'] = trade_ymd
            return df
        
    except Exception as e:
        print(f"日期 {trade_ymd} 特征提取出错: {e}")
    
    return None

In [9]:
# 提取所有日期的特征
print("开始提取特征...")

all_features = []
for trade_ymd in march_dates:
    print(f"处理日期: {trade_ymd}", end='\r')
    features_df = extract_features_for_date(trade_ymd, instrument_id, param_dict)
    if features_df is not None:
        all_features.append(features_df)

print("\n特征提取完成!")

开始提取特征...
处理日期: 20260327
特征提取完成!


In [10]:
# 合并所有特征
if all_features:
    features_df = pd.concat(all_features, ignore_index=True)
    print(f"特征数据形状: {features_df.shape}")
    print(f"特征列: {list(features_df.columns)}")
    
    # 显示前几行
    print("\n前5行数据:")
    print(features_df.head())
else:
    print("没有提取到特征数据!")

特征数据形状: (286840, 16)
特征列: ['num_trades', 'best_bid', 'best_ask', 'volatility', 'spread', 'WAMP', 'alpha_01', 'alpha_02', 'alpha_03', 'alpha_04', 'alpha_05', 'alpha_06', 'alpha_07', 'hurst_exponent', 'time_mark', 'date']

前5行数据:
   num_trades  best_bid  best_ask  volatility    spread       WAMP  alpha_01  alpha_02  alpha_03    alpha_04  alpha_05      alpha_06  alpha_07  \
0         142    11.248    11.254    0.003580  0.000533  11.252792       1.0       1.0  0.202461  200.800000  0.224121  1.418463e-10  0.202461   
1         165    11.232    11.254    0.003528  0.001959  11.241940       1.0       1.0  0.555956  202.216667  0.227861  3.619607e-10  0.555956   
2         122    11.232    11.248    0.003494  0.001425  11.239293       1.0       1.0  0.562017  203.866667  0.248241  2.935020e-10  0.562017   
3         216    11.231    11.240    0.003520  0.000801  11.232613       1.0       1.0  0.564029  205.200000  0.245771  3.989162e-10  0.564029   
4         120    11.238    11.248    0.003

## 5. 创建未来收益数据

In [11]:
def calculate_forward_returns(features_df, forward_periods=60):
    """计算未来收益"""
    # 按日期和时间排序
    features_df = features_df.sort_values(['date', 'time_mark']).reset_index(drop=True)
    
    # 获取价格数据（使用WAMP作为价格代理）
    prices = features_df['WAMP'].values
    
    # 计算未来收益
    forward_returns = np.full(len(prices), np.nan)
    
    for i in range(len(prices) - forward_periods):
        current_price = prices[i]
        future_price = prices[i + forward_periods]
        
        if current_price > 0:
            # 计算对数收益
            forward_returns[i] = np.log(future_price / current_price)
    
    return forward_returns

In [12]:
# 计算未来收益（未来60秒）
if 'features_df' in locals() and not features_df.empty:
    forward_returns = calculate_forward_returns(features_df, forward_periods=60)
    
    # 创建收益Series
    returns_series = pd.Series(
        forward_returns,
        index=pd.MultiIndex.from_arrays(
            [features_df['date'], features_df['time_mark']],
            names=['date', 'time_mark']
        ),
        name='forward_return'
    )
    
    # 移除NaN值
    returns_series = returns_series.dropna()
    
    print(f"未来收益数据形状: {returns_series.shape}")
    print(f"\n收益统计:")
    print(returns_series.describe())
else:
    print("没有特征数据可用!")

未来收益数据形状: (286462,)

收益统计:
count    2.864620e+05
mean             -inf
std               NaN
min              -inf
25%     -2.510611e-04
50%     -6.902800e-06
75%      2.257964e-04
max      4.368198e-02
Name: forward_return, dtype: float64


/tmp/ipykernel_113540/3098264842.py:18: RuntimeWarning: divide by zero encountered in log
  forward_returns[i] = np.log(future_price / current_price)


## 6. 准备因子数据

In [13]:
# 准备因子数据
if 'features_df' in locals() and 'returns_series' in locals():
    # 选择要测试的因子
    factor_columns = [
        'volatility',    # 波动率
        'spread',        # 买卖价差
        'WAMP',          # 加权平均中间价
        'alpha_03',      # 买卖量差
        'alpha_04',      # 交易频率
        'alpha_05',      # 买卖交易次数差
        'hurst_exponent' # 赫斯特指数
    ]
    
    # 创建因子DataFrame
    factor_df = features_df[['date', 'time_mark'] + factor_columns].copy()
    
    # 设置MultiIndex
    factor_df.set_index(['date', 'time_mark'], inplace=True)
    
    # 对齐因子和收益数据
    common_index = factor_df.index.intersection(returns_series.index)
    factor_df = factor_df.loc[common_index]
    returns_series = returns_series.loc[common_index]
    
    print(f"对齐后因子数据形状: {factor_df.shape}")
    print(f"对齐后收益数据形状: {returns_series.shape}")
    
    # 显示因子基本信息
    print("\n因子描述统计:")
    print(factor_df.describe())
    
    # 创建FactorData对象
    factor_data = FactorData(factor_df)
    print(f"\nFactorData创建成功: {factor_data}")
else:
    print("数据准备失败!")

对齐后因子数据形状: (286462, 7)
对齐后收益数据形状: (286462,)

因子描述统计:
         volatility         spread           WAMP       alpha_03       alpha_04       alpha_05  hurst_exponent
count  2.864620e+05  286462.000000  286462.000000  286462.000000  286462.000000  286462.000000   286462.000000
mean   1.672636e-04       0.000108      10.471931      -0.024960      14.867509       0.029021        0.435545
std    1.567446e-04       0.000041       0.678835       0.275298      13.834795       0.091864        0.101486
min    1.994562e-16       0.000088       8.906002      -0.919127       1.466667      -0.400000        0.016585
25%    7.258726e-05       0.000091       9.893602      -0.208085       6.616667      -0.028571        0.369843
50%    1.172084e-04       0.000094      10.797246      -0.024785      10.800000       0.026549        0.435671
75%    2.026893e-04       0.000104      10.963998       0.157235      17.900000       0.085106        0.502329
max    3.579761e-03       0.001959      11.418997       0.9

## 7. 因子预处理

In [14]:
# 因子预处理示例
if 'factor_data' in locals():
    # 选择一个因子进行预处理演示
    test_factor_name = 'volatility'
    test_factor = factor_data.get_factor(test_factor_name)
    
    print(f"原始 {test_factor_name} 因子统计:")
    print(f"  均值: {test_factor.mean():.6f}")
    print(f"  标准差: {test_factor.std():.6f}")
    print(f"  最小值: {test_factor.min():.6f}")
    print(f"  最大值: {test_factor.max():.6f}")
    
    # 去极值处理
    winsorized = FactorPreprocessor.winsorize(
        test_factor, method='quantile', limits=0.05
    )
    
    print(f"\n去极值后 {test_factor_name} 因子统计:")
    print(f"  均值: {winsorized.mean():.6f}")
    print(f"  标准差: {winsorized.std():.6f}")
    print(f"  最小值: {winsorized.min():.6f}")
    print(f"  最大值: {winsorized.max():.6f}")
    
    # 标准化处理
    standardized = FactorPreprocessor.standardize(
        winsorized, method='zscore'
    )
    
    print(f"\n标准化后 {test_factor_name} 因子统计:")
    print(f"  均值: {standardized.mean():.6f}")
    print(f"  标准差: {standardized.std():.6f}")
else:
    print("因子数据不可用!")

原始 volatility 因子统计:
  均值: 0.000167
  标准差: 0.000157
  最小值: 0.000000
  最大值: 0.003580

去极值后 volatility 因子统计:
  均值: 0.000157
  标准差: 0.000111
  最小值: 0.000048
  最大值: 0.000450

标准化后 volatility 因子统计:
  均值: 0.000000
  标准差: 1.000000


## 8. IC分析

In [15]:
# 计算单个因子的IC
if 'factor_data' in locals() and 'returns_series' in locals():
    from factor_testing.metrics import ICCalculator
    
    # 测试波动率因子
    volatility_factor = factor_data.get_factor('volatility')
    
    # 创建IC计算器
    ic_calculator = ICCalculator(volatility_factor, returns_series)
    
    # 计算Pearson IC
    ic_pearson = ic_calculator.calculate_ic(method='pearson')
    print(f"波动率因子 Pearson IC: {ic_pearson:.4f}")
    
    # 计算Rank IC (Spearman)
    ic_spearman = ic_calculator.calculate_ic(method='spearman')
    print(f"波动率因子 Rank IC: {ic_spearman:.4f}")
    
    # 计算IC时间序列（按日期）
    ic_series = ic_calculator.calculate_ic_series(freq='D', method='pearson')
    
    print(f"\nIC时间序列统计:")
    print(f"  均值: {ic_series.mean():.4f}")
    print(f"  标准差: {ic_series.std():.4f}")
    print(f"  IR: {ic_series.mean() / ic_series.std() if ic_series.std() != 0 else np.nan:.4f}")
    print(f"  正IC比例: {(ic_series > 0).mean() * 100:.1f}%")
else:
    print("数据不可用!")

/opt/conda/lib/python3.13/site-packages/scipy/stats/_stats_py.py:4735: RuntimeWarning: invalid value encountered in subtract
  ym = y - ymean


波动率因子 Pearson IC: nan
波动率因子 Rank IC: 0.0016


/opt/conda/lib/python3.13/site-packages/scipy/stats/_stats_py.py:4735: RuntimeWarning: invalid value encountered in subtract
  ym = y - ymean



IC时间序列统计:
  均值: 0.0409
  标准差: 0.0540
  IR: 0.7564
  正IC比例: 85.0%


## 9. 综合指标计算

In [16]:
# 计算因子的所有指标
if 'factor_data' in locals() and 'returns_series' in locals():
    # 创建因子指标计算器
    metrics_calculator = FactorMetrics(
        factor_data.get_factor('volatility'),
        returns_series
    )
    
    # 计算所有指标
    all_metrics = metrics_calculator.calculate_all_metrics(
        n_groups=5, freq='D', method='pearson'
    )
    
    print("波动率因子综合指标:")
    print("=" * 50)
    
    # 显示关键指标
    key_metrics = {
        'IC': all_metrics.get('ic', np.nan),
        'Rank IC': all_metrics.get('rank_ic', np.nan),
        'IR': all_metrics.get('ir', np.nan),
        'IC均值': all_metrics.get('ic_mean', np.nan),
        'IC标准差': all_metrics.get('ic_std', np.nan),
        '正IC比例': all_metrics.get('ic_positive_rate', np.nan) * 100,
        '多空组合夏普': all_metrics.get('long_short_sharpe', np.nan),
        '多空组合收益': all_metrics.get('long_short_mean_return', np.nan) * 100,
        '平均换手率': all_metrics.get('avg_turnover', np.nan) * 100
    }
    
    for metric_name, value in key_metrics.items():
        if pd.notnull(value):
            if '比例' in metric_name or '收益' in metric_name or '换手率' in metric_name:
                print(f"  {metric_name}: {value:.2f}%")
            else:
                print(f"  {metric_name}: {value:.4f}")
else:
    print("数据不可用!")

/opt/conda/lib/python3.13/site-packages/scipy/stats/_stats_py.py:4735: RuntimeWarning: invalid value encountered in subtract
  ym = y - ymean
/opt/conda/lib/python3.13/site-packages/scipy/stats/_stats_py.py:4735: RuntimeWarning: invalid value encountered in subtract
  ym = y - ymean
/opt/conda/lib/python3.13/site-packages/scipy/stats/_stats_py.py:4735: RuntimeWarning: invalid value encountered in subtract
  ym = y - ymean
/opt/conda/lib/python3.13/site-packages/scipy/stats/_stats_py.py:4735: RuntimeWarning: invalid value encountered in subtract
  ym = y - ymean
/opt/conda/lib/python3.13/site-packages/scipy/stats/_stats_py.py:4735: RuntimeWarning: invalid value encountered in subtract
  ym = y - ymean
/opt/conda/lib/python3.13/site-packages/scipy/stats/_stats_py.py:4735: RuntimeWarning: invalid value encountered in subtract
  ym = y - ymean
/opt/conda/lib/python3.13/site-packages/scipy/stats/_stats_py.py:4735: RuntimeWarning: invalid value encountered in subtract
  ym = y - ymean


波动率因子综合指标:
  Rank IC: 0.0016
  IR: 0.7564
  多空组合夏普: -0.1094
  多空组合收益: -0.00%
  平均换手率: 100.00%


/home/jovyan/work/tactics_demo/factor_testing/metrics/factor_metrics.py:329: RuntimeWarning: invalid value encountered in scalar subtract
  group_returns["long_short"] = long_return - short_return


## 10. 批量计算多个因子

In [17]:
# 批量计算所有因子的指标
if 'factor_data' in locals() and 'returns_series' in locals():
    # 获取所有因子数据
    all_factors_df = factor_data.get_factors()
    
    # 批量计算指标
    batch_metrics = FactorMetrics.batch_calculate_metrics(
        all_factors_df, returns_series, n_groups=5, freq='D', method='pearson'
    )
    
    print("所有因子指标比较:")
    print("=" * 60)
    
    # 先打印列名以了解数据结构
    print(f"batch_metrics列名: {list(batch_metrics.columns)}")
    
    # 根据实际列名选择关键指标显示
    # 注意：根据factor_metrics.py，IC相关指标可能有'ic_'前缀
    available_columns = list(batch_metrics.columns)
    
    # 尝试找到正确的列名
    ic_column = 'ic' if 'ic' in available_columns else ('ic_ic' if 'ic_ic' in available_columns else None)
    ir_column = 'ir' if 'ir' in available_columns else None
    rank_ic_column = 'rank_ic' if 'rank_ic' in available_columns else None
    long_short_sharpe_column = 'long_short_sharpe' if 'long_short_sharpe' in available_columns else None
    long_short_mean_return_column = 'long_short_mean_return' if 'long_short_mean_return' in available_columns else None
    avg_turnover_column = 'avg_turnover' if 'avg_turnover' in available_columns else None
    
    # 构建可用的指标列表
    key_metrics = []
    column_mapping = {}
    
    if ic_column:
        key_metrics.append(ic_column)
        column_mapping[ic_column] = 'IC'
    if ir_column:
        key_metrics.append(ir_column)
        column_mapping[ir_column] = 'IR'
    if rank_ic_column:
        key_metrics.append(rank_ic_column)
        column_mapping[rank_ic_column] = 'Rank IC'
    if long_short_sharpe_column:
        key_metrics.append(long_short_sharpe_column)
        column_mapping[long_short_sharpe_column] = '多空夏普'
    if long_short_mean_return_column:
        key_metrics.append(long_short_mean_return_column)
        column_mapping[long_short_mean_return_column] = '多空收益'
    if avg_turnover_column:
        key_metrics.append(avg_turnover_column)
        column_mapping[avg_turnover_column] = '平均换手率'
    
    if key_metrics:
        display_df = batch_metrics[key_metrics].copy()
        # 重命名列以便更好显示
        display_df = display_df.rename(columns=column_mapping)
    else:
        print("警告: 未找到任何关键指标列，显示所有列")
        display_df = batch_metrics.copy()
    
    # 格式化显示
    for col in display_df.columns:
        if col in ['IC', 'IR', 'Rank IC', '多空夏普']:
            display_df[col] = display_df[col].apply(lambda x: f"{x:.4f}" if pd.notnull(x) else "NaN")
        elif col == '多空收益':
            display_df[col] = display_df[col].apply(lambda x: f"{x*100:.2f}%" if pd.notnull(x) else "NaN")
        elif col == '平均换手率':
            display_df[col] = display_df[col].apply(lambda x: f"{x*100:.1f}%" if pd.notnull(x) else "NaN")
    
    print(display_df)
    
    # 按IR排序
    print("\n按IR排序:")
    print("-" * 60)
    if ir_column and ir_column in batch_metrics.columns:
        ir_sorted = batch_metrics.sort_values(ir_column, ascending=False)[ir_column]
        for factor, ir in ir_sorted.items():
            print(f"  {factor:20}: {ir:.4f}")
    else:
        print("  未找到IR列")
else:
    print("数据不可用!")

/opt/conda/lib/python3.13/site-packages/scipy/stats/_stats_py.py:4735: RuntimeWarning: invalid value encountered in subtract
  ym = y - ymean
/opt/conda/lib/python3.13/site-packages/scipy/stats/_stats_py.py:4735: RuntimeWarning: invalid value encountered in subtract
  ym = y - ymean
/opt/conda/lib/python3.13/site-packages/scipy/stats/_stats_py.py:4735: RuntimeWarning: invalid value encountered in subtract
  ym = y - ymean
/opt/conda/lib/python3.13/site-packages/scipy/stats/_stats_py.py:4735: RuntimeWarning: invalid value encountered in subtract
  ym = y - ymean
/opt/conda/lib/python3.13/site-packages/scipy/stats/_stats_py.py:4735: RuntimeWarning: invalid value encountered in subtract
  ym = y - ymean
/opt/conda/lib/python3.13/site-packages/scipy/stats/_stats_py.py:4735: RuntimeWarning: invalid value encountered in subtract
  ym = y - ymean
/opt/conda/lib/python3.13/site-packages/scipy/stats/_stats_py.py:4735: RuntimeWarning: invalid value encountered in subtract
  ym = y - ymean
/home/

所有因子指标比较:


KeyError: "['ic'] not in index"

## 11. 分组测试

In [ ]:
# 对最佳因子进行分组测试
if 'factor_data' in locals() and 'returns_series' in locals() and 'batch_metrics' in locals():
    # 选择IR最高的因子
    # 先确定IR列名
    available_columns = list(batch_metrics.columns)
    ir_column = 'ir' if 'ir' in available_columns else None
    
    if ir_column and ir_column in batch_metrics.columns:
        best_factor_name = batch_metrics[ir_column].idxmax() if not batch_metrics.empty else 'volatility'
    else:
        print("警告: 未找到IR列，使用默认因子'volatility'")
        best_factor_name = 'volatility'
    
    best_factor = factor_data.get_factor(best_factor_name)
    
    print(f"对最佳因子 '{best_factor_name}' 进行分组测试:")
    print("=" * 60)
    
    # 创建分组测试器
    group_tester = GroupTester(best_factor, returns_series)
    
    # 运行全面测试
    test_results = group_tester.run_comprehensive_test(
        n_groups=5, method='quantile', rebalance_freq='D'
    )
    
    # 显示多空组合表现
    if 'long_short' in test_results:
        ls = test_results['long_short']
        print("\n多空组合表现:")
        print("-" * 40)
        print(f"  平均收益: {ls.get('mean_return', 0)*100:.2f}%")
        print(f"  夏普比率: {ls.get('sharpe_ratio', 0):.4f}")
        print(f"  胜率: {ls.get('win_rate', 0)*100:.1f}%")
    
    # 显示单调性
    if 'monotonicity' in test_results:
        mono = test_results['monotonicity']
        print(f"\n分组单调性 (Spearman相关系数): {mono.get('spearman_corr', 0):.4f}")
else:
    print("数据不可用!")

## 12. 生成报告

In [ ]:
# 生成因子分析报告
if 'factor_data' in locals() and 'returns_series' in locals():
    # 选择要生成报告的因子
    report_factor_name = 'volatility'
    if 'best_factor_name' in locals():
        report_factor_name = best_factor_name
    
    report_factor = factor_data.get_factor(report_factor_name)
    
    print(f"生成 '{report_factor_name}' 因子分析报告...")
    
    # 创建报告生成器
    report_gen = ReportGenerator(
        factor_name=report_factor_name,
        factor_data=report_factor,
        forward_returns=returns_series
    )
    
    # 生成文本摘要报告
    report_text = report_gen.generate_summary_report(
        n_groups=5, method='quantile', freq='D', ic_method='pearson'
    )
    
    print("\n因子分析报告摘要:")
    print("=" * 80)
    print(report_text[:500] + "..." if len(report_text) > 500 else report_text)
    
    # 生成图表
    print("\n生成分析图表...")
    
    # 因子分布图
    fig1 = report_gen.generate_factor_distribution_plot()
    plt.suptitle(f"{report_factor_name} 因子分布分析", fontsize=14, fontweight='bold')
    plt.show()
    
    # IC分析图
    fig2 = report_gen.generate_ic_analysis_plot(freq='D', method='pearson')
    plt.suptitle(f"{report_factor_name} IC分析", fontsize=14, fontweight='bold')
    plt.show()
    
    # 分组表现图
    fig3 = report_gen.generate_group_performance_plot(n_groups=5, method='quantile')
    plt.suptitle(f"{report_factor_name} 分组表现分析", fontsize=14, fontweight='bold')
    plt.show()
    
    # 保存报告
    output_dir = f"./factor_analysis_report_{report_factor_name}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    print(f"\n保存完整报告到: {output_dir}")
    report_gen.save_report(
        output_dir=output_dir,
        n_groups=5,
        method='quantile',
        freq='D',
        ic_method='pearson'
    )
    
    print("\n分析完成!")
    print(f"分析报告已保存到: {output_dir}")
else:
    print("数据不可用!")